In [1]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# xx_cat_spec_investigation.ipynb
# Purpose of Script: Assess Coverage of Category Specification
# Media
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initialization ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Google Drive
#~~~~~~~~~~~~~~~~~~~~~~~~~~
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Libraries
#~~~~~~~~~~~~~~~~~~~~~~~~~~
import numpy as np
import pandas as pd
import duckdb
from pathlib import Path

In [3]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Print Versions
#~~~~~~~~~~~~~~~~~~~~~~~~~~
print(f"Numpy version = {np.__version__}")
print(f"Pandas version = {pd.__version__}")
print(f"DuckDB version = {duckdb.__version__}")

Numpy version = 2.0.2
Pandas version = 2.2.2
DuckDB version = 1.3.2


In [4]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initiate Duck Connection
#~~~~~~~~~~~~~~~~~~~~~~~~~~
con = duckdb.connect()

#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Define Input/Output Paths
#~~~~~~~~~~~~~~~~~~~~~~~~~~
### Input
path_samp = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/03_samples/"
path_clean = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/02_clean/"
path_out = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/03_outputs/"
path_dup = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/05_duplicates/"


In [20]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Assess Cat Spec Column
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Define Base Directory
dir_base = Path(path_clean)

# List All Files
files_facebook = sorted(dir_base.glob("*-facebook/facebook.parquet"))
files_instagram = sorted(dir_base.glob("*-instagram/instagram.parquet"))
files_tiktok = sorted(dir_base.glob("*-tiktok/tiktok.parquet"))
files_youtube = sorted(dir_base.glob("*-youtube/youtube.parquet"))
files_x = sorted(dir_base.glob("*-x/x.parquet"))
files_whatsapp = sorted(dir_base.glob("*-whatsapp/whatsapp.parquet"))
files_snapchat = sorted(dir_base.glob("*-snapchat/snapchat.parquet"))

# Cut Down Files
files_facebook = files_facebook[1:10]
files_instagram = files_instagram[1:10]
files_tiktok = files_tiktok[1:10]
files_youtube = files_youtube[1:10]
files_x = files_x[1:10]
files_whatsapp = files_whatsapp[1:10]
files_snapchat = files_snapchat[1:10]

In [21]:
# Sample All Files
samples = []

# All Files
files = {"facebook": files_facebook, "instagram" : files_instagram,
         "tiktok" : files_tiktok, "youtube": files_youtube,
         "x" : files_x, "whatsapp" : files_whatsapp,
         "snapchat" : files_snapchat}

# Start File
file_num = 1

# Loop
for platform, platform_files in files.items():

    # Print Progress
    print(f"Processing Platform: {platform}")

    # Drop Temporary Table
    con.execute("drop table if exists sample")

    # Column Selection
    if platform in ["facebook", "instagram"]:
        cols_to_select = """ p_name, date, cat_spec """
    else:
        cols_to_select = """ p_name, date, cat_spec """

    first = True

    for file in platform_files:

        # Print Progress
        print(f"Processing File: {file}")

        if first:

            con.execute(f""" create table sample as
                             select p_name as platform, date, cat_spec, count(*)
                             from read_parquet('{file}')
                             group by p_name, date, cat_spec""")

            first = False

        else:

            con.execute(f""" insert into sample
                             select p_name as platform, date, cat_spec, count(*)
                             from read_parquet('{file}')
                             group by p_name, date, cat_spec""")

    con.execute(f""" copy sample
                     to '{path_out}xx_cat_spec_{file_num}_{platform}.parquet'
                     (format parquet)""")

    # Increase File Number
    file_num += 1

    # Clean Results
    con.execute(f""" drop table sample""")
    con.commit()
    con.close()
    con = duckdb.connect("/content/working.duckdb")

Processing Platform: facebook
Processing File: /content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/02_clean/2025-01-02-facebook/facebook.parquet
Processing File: /content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/02_clean/2025-01-03-facebook/facebook.parquet
Processing File: /content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/02_clean/2025-01-04-facebook/facebook.parquet
Processing File: /content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/02_clean/2025-01-05-facebook/facebook.parquet
Processing File: /content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/02_clean/2025-01-06-facebook/facebook.parquet
Processing File: /content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/02_clean/2025-01-07-facebook/facebook.parquet
Processing File: /content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/02_clean/2025-01-08-facebook/facebook.parquet
Processing 